### Importing Libraries

In [ ]:
import math
import torch
import tiktoken
import torch.nn as nn
from google.colab import drive
from torch.nn import functional as F
from transformers import AutoTokenizer
from torch.optim.lr_scheduler import CosineAnnealingLR

drive.mount('/content/drive/')

### Loading the dataset

In [ ]:
with open('/content/drive/MyDrive/data/train.csv', 'r', encoding='utf-8') as f:
    text = f.read()


#### Explore the dataset

In [ ]:
print("The length of the dataset is: ", len(text))
print(text[:1000])

## Tokenization Layer

In [ ]:
# encoding = tiktoken.get_encoding("o200k_base")
encoding = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased",
)

tokens = encoding.encode(text)
vocab_size = len(tokens)
print(vocab_size)
print(len(tokens))
print(tokens[:10])

In [ ]:
data = torch.tensor(tokens, dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

#### Split up the data into train and validation sets

In [ ]:
n = int(0.9*len(data)) # first 80% will be train, rest val
train_data = data[:n]
val_data = data[n:]

train_data[:10]

### Define Hyperparameters

In [ ]:
batch_size = 16
block_size = 128
max_iters = 10000
eval_interval = 500
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 8
n_layer = 8
dropout = 0

#### generate batch of data X and Y

In [ ]:
def get_batch(split):
    data = train_data if split == 'train' else val_data

    ''' get a random number(index) and take a context
    window starting from there ( random context
    windows will help the model to learn better )'''

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [ ]:
x, y = get_batch('train')
print("when the input context window is like this: \n")
print(encoding.decode(x[1].tolist()))
print("\n\nthe out put will be like this:\n")
print(encoding.decode(y[1].tolist()))

### Implementing a layer to calculate loss

In [ ]:
@torch.no_grad()  # this stops the gradual descent ( it won't try to run gradual descent to update parameters )
def estimate_loss():
    out = {}
    model.eval()   # evaluation mode insures the model is not trying to learn( it stops dropout and BatchNorm )
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split) # select random batch to evaluate the model with
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### Implementing a single attention head layer

In [ ]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        # a neural layer which transforms the input vector into of dimension (B, T, C) -> (B, T, head_size) 
        # key explains the tokens dentity(itself) / how it will be looked up by others
        self.key = nn.Linear(n_embd, head_size, bias=False)
        # query explains what the token is looking for
        self.query = nn.Linear(n_embd, head_size, bias=False)
        # value is the actual value of the token
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        ''' B -> Batch size(16)
            T -> Time(32) (Block Size) (context window size/ sequence length)
            C -> number of embedding for each token
        '''
        B,T,C = x.shape

        k = self.key(x)   # (B, T, C) -> (B, T, head_size)
        q = self.query(x) # (B, T, C) -> (B, T, head_size)

        # compute attention scores (how much token i matches token j)
        attention_score = q @ k.transpose(-2,-1) * C**-0.5
        attention_score = attention_score.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attention_score = F.softmax(attention_score, dim=-1)
        attention_score = self.dropout(attention_score)

        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, C) -> (B, T, head_size)
        attention = attention_score @ v
        return attention

### Implement the multi Head Attention Layer

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # projects the result of each head into a linear layer
        self.projection = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
      # concatnate the result of each attention head
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # the result of each head is mixed
        out = self.dropout(self.projection(out))
        return out

### Implement a nural layer that introduces non-linearity

In [ ]:
class FeedFoward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # linear layer which maps the input to each head (B, T, C) -> (B, T, c*4)
            # enables it to consider complex token features
            nn.Linear(n_embd, 4 * n_embd),
            # non-linear layer( activation function )
            nn.ReLU(),
            # linear layer which maps the head to input dimension (B, T, C*4) -> (B, T, C)
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## Transformer Layer

In [ ]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads
        super().__init__()
        head_size = n_embd // n_head
        # calling multihead attention to get attention scores
        self.mHA = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        # apply normalization
        self.linearLayer1 = nn.LayerNorm(n_embd)
        self.linearLayer2 = nn.LayerNorm(n_embd)

    def forward(self, x):
      # residual connection
        x = x + self.mHA(self.linearLayer1(x))
        x = x + self.ffwd(self.linearLayer2(x))
        return x

### Decode Only Tokenizer

In [ ]:
# super simple bigram model
class DecodOnlyTransformer(nn.Module):

    def __init__(self):
        super().__init__()
        # token embedding( Context wise )
        # projects vocab_size into n_embd dimension
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        # position embedding( Position wise )
        self.position_embedding = nn.Embedding(block_size, n_embd)

        # call the transformer block n_layer times
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.layernorm_f = nn.LayerNorm(n_embd)

        # projects the n_embd dimension to vocab_size dimension
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, token, targets=None):
        B, T = token.shape
        # token and targets are both (B,T) tensor of integers
        token_emb = self.token_embedding(token)
        position_emb = self.position_embedding(torch.arange(T, device=device)) # (T,C)
        x = token_emb + position_emb # (B,T,C)

        # this turns raw embeddings (x) into contextulized vector
        x = self.blocks(x) # (B,T,C)
        x = self.layernorm_f(x) # (B,T,C)

        # returns the prediction for the next token
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None: # if we are not training then the loss is not needed
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, token, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop token to the last block_size tokens
            token_cond = token[:, -block_size:]
            # get the predictions
            logits, loss = self(token_cond)
            # focus only on the last time step
            logits = logits[:, -1, :]
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            token_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled token to the running sequence
            token = torch.cat((token, token_next), dim=1) # (B, T+1)
        return token


## Train the model

In [ ]:
model = DecodOnlyTransformer()
m = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
scheduler = CosineAnnealingLR(optimizer, T_max=max_iters)

best_val_loss = float('inf')

for iter in range(max_iters):

    if iter % 500 == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_val_loss,
            }, '/content/drive/MyDrive/data/decoder_model3.pth')
            print(f"Saved at step {iter} | best val loss: {best_val_loss:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    scheduler.step()

### Save the model

In [ ]:
PATH = '/content/drive/MyDrive/data/decoder_model3.pth'

checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}


torch.save(checkpoint, PATH)

### Evaluate the model

In [ ]:
def evaluate_model():
    checkpoint = torch.load(PATH, map_location=device)
    model = DecodOnlyTransformer()
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()

    total_loss = 0
    total_tokens = 0
    num_batches = 0

    with torch.no_grad():
        for _ in range(100):
            x, y = get_batch('val')
            logits, loss = model(x, y)

            if loss is not None:
                total_loss += loss.item() * x.numel()
                total_tokens += x.numel()
                num_batches += 1

    if total_tokens > 0:
        avg_loss = total_loss / total_tokens
        perplexity = math.exp(avg_loss)
    else:
        avg_loss = float('inf')
        perplexity = float('inf')

    print(f"Evaluation completed:")
    print(f"Average Loss: {avg_loss:.4f}")
    print(f"Perplexity: {perplexity:.4f}")

    return {
        'perplexity': perplexity,
        'loss': avg_loss,
        'note': f'Evaluated on {num_batches} batches, {total_tokens} tokens'
    }

In [ ]:
evaluate_model()